# Notebook 01 — Signal Acquisition

This notebook covers Layer 1 of the CAPS Framework: identifying, loading,
validating, and structuring the behavioral signal matrix from a competitive
product launch event dataset.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from caps.acquisition.loaders import _generate_synthetic_data, load_from_csv
from caps.acquisition.signal_matrix import SignalMatrix, REQUIRED_COLUMNS
from caps.acquisition.validators import SignalValidator

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'serif'

## 1.1 — Load Dataset

We generate a synthetic dataset of 500 consumers observed across a 90-day
window following a competitive product launch. The `true_segment` column
is included for validation only and is not used by the pipeline.

In [ ]:
df = _generate_synthetic_data(n_consumers=500, random_state=42)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(8)

## 1.2 — Descriptive Statistics

In [ ]:
signal_cols = [
    'adoption_lag_days', 'brand_switch', 'price_elasticity',
    'attribute_priority_score', 'loyalty_retention_ratio',
    'category_spend_share', 'trial_frequency'
]
df[signal_cols].describe().round(4)

## 1.3 — Signal Distribution Plots

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

labels_map = {
    'adoption_lag_days': 'Adoption Lag (days)',
    'brand_switch': 'Brand Switch (0/1)',
    'price_elasticity': 'Price Elasticity',
    'attribute_priority_score': 'Attribute Priority Score',
    'loyalty_retention_ratio': 'Loyalty Retention Ratio',
    'category_spend_share': 'Category Spend Share',
    'trial_frequency': 'Trial Frequency',
}

segment_colors = {
    'High-Power Adopters': '#1a5276',
    'Flexible Mid-Tier': '#2e86c1',
    'Price-Opportunist': '#aed6f1',
    'Disengaged Low-Power': '#99a3a4',
}

for i, col in enumerate(signal_cols):
    ax = axes[i]
    for seg, color in segment_colors.items():
        subset = df[df['true_segment'] == seg][col]
        if col == 'brand_switch':
            ax.bar(
                [seg[:6]], [subset.mean()], color=color, alpha=0.8
            )
        else:
            subset.plot.kde(ax=ax, color=color, linewidth=1.8, label=seg[:12])
    ax.set_title(labels_map[col], fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if i == 0:
        ax.legend(fontsize=6)

axes[-1].set_visible(False)
plt.suptitle('Signal Dimension Distributions by True Segment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 1.4 — Correlation Matrix of Signal Dimensions

In [ ]:
corr = df[signal_cols].corr().round(3)
fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, ax=ax, mask=mask, annot=True, fmt='.3f',
    cmap='RdBu_r', center=0, linewidths=0.5,
    xticklabels=[l[:12] for l in labels_map.values()],
    yticklabels=[l[:12] for l in labels_map.values()],
    cbar_kws={'label': 'Pearson r'}
)
ax.set_title('Signal Dimension Correlation Matrix', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 1.5 — Build and Validate Signal Matrix

In [ ]:
class _Cfg:
    observation_window = 90
    switching_threshold = 0.30

sm = SignalMatrix(_Cfg())
X, consumer_ids = sm.build(df)
print(f'Signal matrix shape: {X.shape}')
print(f'Consumer count: {len(consumer_ids)}')
print(f'Signal dimensions: {X.shape[1]}')

validator = SignalValidator(_Cfg())
validator.validate(X)
print('Validation passed.')

## 1.6 — Signal-to-Noise Ratio per Dimension

In [ ]:
col_means = np.abs(X.mean(axis=0))
col_stds = X.std(axis=0)
snr = col_means / (col_stds + 1e-9)

dim_names = [
    'Adoption Lag', 'Brand Switch', 'Price Elasticity',
    'Attr. Priority', 'Loyalty Retention', 'Category Spend', 'Trial Freq.'
][:X.shape[1]]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(dim_names, snr, color=['#1a5276' if s >= 0.65 else '#e74c3c' for s in snr], edgecolor='white')
ax.axhline(0.65, color='black', linestyle='--', linewidth=1.2, label='SNR Threshold (0.65)')
ax.set_ylabel('Signal-to-Noise Ratio', fontsize=10)
ax.set_title('SNR per Signal Dimension — Layer 1 Validation', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for bar, s in zip(bars, snr):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{s:.3f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()